# Data Mining

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from src.categories import tp_level_categories
from src.descriptions import df_measurements, df_descriptions
from src.eda import understand_features
from src.enums import files, keys, prefixes
from src.serialize import dump_to_file

df = pd.read_csv(files['data/04-03-dataset.csv'])

## Understand features

In [2]:
understand_features(df, df_measurements, df_descriptions)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   RSV_TP_LEVEL              138 non-null    object 
 1   nitrogen_dioxide (μg/m³)  138 non-null    float64
 2   carbon_monoxide (μg/m³)   138 non-null    float64
 3   uv_index ()               138 non-null    float64
 4   ozone (μg/m³)             138 non-null    float64
 5   MMWR_WEEK                 138 non-null    int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 6.6+ KB


,Measurement Type,Description
nitrogen_dioxide (μg/m³),ratio,Vehicle emissions can cause airway inflammation.
carbon_monoxide (μg/m³),ratio,"Odorless gas from combustion, reduces oxygen d..."
uv_index (),ratio,Measurement of the intensity of ultraviolet ra...
ozone (μg/m³),ratio,Created by chemical reactions between oxides o...
RSV_TP_LEVEL,ordinal,Positivity level based on 5 seasons of RSV data.
MMWR_WEEK,interval,Standardized CDC week number (1 to 52 or 53).


## Data Preparation

### Discretize target variable

In [3]:
df[keys['RSV_TP_LEVEL']] = pd.Categorical(
    df[keys['RSV_TP_LEVEL']],
    categories=tp_level_categories,
    ordered=True
)
df[keys['RSV_TP_LEVEL']] = df[keys['RSV_TP_LEVEL']].cat.codes

### Split dataset

In [4]:
target = df[keys['RSV_TP_LEVEL']]
features = df.drop(columns=['RSV_TP_LEVEL'])
n_splits = 5
time_series_split = TimeSeriesSplit(n_splits=n_splits)

folds = list(time_series_split.split(features))

for index, (train_index, test_index) in enumerate(folds):
    # Serialize the training folds to allow analysis in 04-03-data-mining-analyzer notebook.
    dump_to_file(df.iloc[train_index], prefixes['04_03_data_mining_folds_'] + str(index))

validation_folds = folds[0:n_splits-1]
test_fold = folds[-1]

## Train and validate model

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    min_samples_leaf=3,
    min_samples_split=2,
    random_state=42
)

## Test model